# Biblioteca

In [ ]:
import os
# TF_USE_LEGACY_KERAS precisa vir antes do import do TensorFlow
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import unicodedata
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

BASE = os.getcwd()
print('BASE:', BASE)
print('TensorFlow:', tf.__version__)

# Modelo e acurácia

In [ ]:
MODELO_DIR = os.path.join(BASE, 'TrashNet', 'Modelo')
TESTE_DIR = os.path.join(BASE, 'TrashNet', 'Dataset', 'ImagemTeste')

# 1) Modelo
model = tf.keras.models.load_model(os.path.join(MODELO_DIR, 'keras_model.h5'), compile=False)

# 2) Classes (ordem do labels.txt = ordem de saída do modelo)
def normaliza(texto):
    sem_acento = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    return sem_acento.strip().lower()

PARA_PASTA = {'metal': 'metal', 'plastico': 'plastic', 'papelao': 'cardboard'}

classes = []  # nome da pasta correspondente, na ordem do modelo
with open(os.path.join(MODELO_DIR, 'labels.txt'), encoding='utf-8') as f:
    for linha in f:
        if linha.strip():
            _, nome = linha.strip().split(' ', 1)
            classes.append(PARA_PASTA[normaliza(nome)])
print('Classes (ordem do modelo):', classes)

# 3) Pré-processamento
def prepara(caminho):
    img = Image.open(caminho).convert('RGB').resize((224, 224))
    arr = np.asarray(img, dtype=np.float32) / 127.5 - 1
    return np.expand_dims(arr, axis=0)

# 4) Predição
EXT = ('.jpg', '.jpeg', '.png', '.bmp')
linhas = []
for pasta in sorted(os.listdir(TESTE_DIR)):
    caminho_pasta = os.path.join(TESTE_DIR, pasta)
    if not os.path.isdir(caminho_pasta):
        continue
    real = pasta.split('_')[0]  # cardboard_20 -> cardboard
    for arquivo in sorted(os.listdir(caminho_pasta)):
        if not arquivo.lower().endswith(EXT):
            continue
        probs = model.predict(prepara(os.path.join(caminho_pasta, arquivo)), verbose=0)[0]
        idx = int(np.argmax(probs))
        prevista = classes[idx]
        linhas.append({
            'arquivo': arquivo,
            'real': real,
            'prevista': prevista,
            'acertou': prevista == real,
            'confianca': round(float(probs[idx]), 3),
        })

df = pd.DataFrame(linhas)

# 5) Resultados
estilo = [{'selector': 'th, td', 'props': [('font-size', '18px'), ('padding', '8px 16px')]}]
por_classe = df.groupby('real').agg(testadas=('acertou', 'size'), acertos=('acertou', 'sum'))
por_classe['acuracia_%'] = (por_classe['acertos'] / por_classe['testadas'] * 100).round(2)
print('Acurácia por classe:')
display(por_classe.style.set_table_styles(estilo))

acuracia_geral = df['acertou'].mean() * 100
print(f'Acurácia geral: {acuracia_geral:.2f}% ({int(df["acertou"].sum())}/{len(df)})')

erros = df.loc[~df['acertou'], ['arquivo', 'real', 'prevista', 'confianca']]
print(f'Erros ({len(erros)}):')
display(df[~df['acertou']][['arquivo', 'real', 'prevista', 'confianca']].style.set_table_styles(estilo).hide(axis='index'))

df.to_csv(os.path.join(BASE, 'resultados_teste.csv'), index=False)
print('CSV salvo em resultados_teste.csv')

# Teste com imagem aleatória

In [ ]:
# Nomes em português, na ordem do modelo (labels.txt)
with open(os.path.join(MODELO_DIR, 'labels.txt'), encoding='utf-8') as f:
    nomes_pt = [linha.strip().split(' ', 1)[1] for linha in f if linha.strip()]

PASTA_PARA_PT = {'metal': 'Metal', 'plastic': 'Plastico', 'cardboard': 'Papelao'}

# Sorteio da pasta e da imagem
pasta_sorteada = random.choice(sorted(
    p for p in os.listdir(TESTE_DIR) if os.path.isdir(os.path.join(TESTE_DIR, p))))
arquivo_sorteado = random.choice(sorted(
    a for a in os.listdir(os.path.join(TESTE_DIR, pasta_sorteada))
    if a.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))))
caminho_img = os.path.join(TESTE_DIR, pasta_sorteada, arquivo_sorteado)

real_pt = PASTA_PARA_PT[pasta_sorteada.split('_')[0]]

# Classificação (usa prepara e model já existentes)
probs = model.predict(prepara(caminho_img), verbose=0)[0]
idx = int(np.argmax(probs))
prevista_pt = nomes_pt[idx]

print('Arquivo:', arquivo_sorteado)
print('Pasta:', pasta_sorteada)
print(f'Classificação: {prevista_pt} ({probs[idx] * 100:.1f}%)')
for nome, p in zip(nomes_pt, probs):
    print(f'  {nome}: {p * 100:.1f}%')

plt.imshow(Image.open(caminho_img).convert('RGB'))
plt.axis('off')
plt.title(f'Real: {real_pt} | Previsto: {prevista_pt}')
plt.show()